# Spam Detection using SVM — Improved Notebook

**Improvements over baseline:**
- Text cleaning (URLs → `url`, numbers → `num`, punctuation removal)
- 7 hand-crafted meta features (message length, URL presence, currency words, caps ratio, etc.)
- TF-IDF upgraded: `max_features=8000`, trigrams, `sublinear_tf=True`
- `class_weight='balanced'` to handle the 87/13 class imbalance
- Model comparison: SVM vs Logistic Regression vs Naive Bayes
- 5-fold cross-validation for all models
- Best model + vectorizer + scaler saved with `joblib`
- `model_results.json` exported for the Streamlit UI

In [ ]:
import pandas as pd
import numpy as np
import re
import json
import joblib
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                              recall_score, classification_report, confusion_matrix)
from scipy.sparse import hstack, csr_matrix

print("All imports successful")

## 1. Load & Inspect Data

In [ ]:
df = pd.read_csv("spam.csv", encoding="latin-1")[['v1', 'v2']]
df.dropna(inplace=True)
df.rename(columns={'v1': 'label', 'v2': 'message'}, inplace=True)

print("Shape:", df.shape)
print("\nClass distribution:")
print(df['label'].value_counts())
print(f"\nSpam %: {df['label'].value_counts(normalize=True)['spam']*100:.1f}%")
df.head()

## 2. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

sns.countplot(x='label', data=df, ax=axes[0], palette='Set2')
axes[0].set_title("Message Type Distribution")
axes[0].set_xlabel("")

df['msg_len_tmp'] = df['message'].str.len()
df.boxplot(column='msg_len_tmp', by='label', ax=axes[1])
axes[1].set_title("Message Length by Class")
axes[1].set_xlabel("")

df['wc_tmp'] = df['message'].str.split().str.len()
df.boxplot(column='wc_tmp', by='label', ax=axes[2])
axes[2].set_title("Word Count by Class")
axes[2].set_xlabel("")

plt.suptitle("")
plt.tight_layout()
plt.show()
df.drop(columns=['msg_len_tmp','wc_tmp'], inplace=True)
print("Avg length — Ham:", df[df['label']=='ham']['message'].str.len().mean().round(1),
      "| Spam:", df[df['label']=='spam']['message'].str.len().mean().round(1))

## 3. Label Encoding

In [ ]:
df['label'] = df['label'].map({'ham': 0, 'spam': 1})
print("Label encoding done: ham=0, spam=1")
df['label'].value_counts()

## 4. Text Cleaning

Baseline only did lowercase + strip. We also:
- Replace URLs with `url` token
- Replace numbers with `num` token
- Remove punctuation
- Collapse whitespace

In [ ]:
def clean_text(text):
    text = text.lower().strip()
    text = re.sub(r'http\S+|www\S+', ' url ', text)
    text = re.sub(r'\b\d+\b', ' num ', text)
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['message_clean'] = df['message'].apply(clean_text)

print("Original :", df['message'].iloc[2])
print("Cleaned  :", df['message_clean'].iloc[2])

## 5. Hand-Crafted Meta Features

These features capture signal that TF-IDF misses.

In [ ]:
META_COLS = ['msg_len','word_count','has_url','has_number',
             'has_currency','caps_ratio','exclaim_count']

def add_meta_features(df):
    df = df.copy()
    df['msg_len']       = df['message'].str.len()
    df['word_count']    = df['message'].str.split().str.len()
    df['has_url']       = df['message'].str.contains(
                            r'http|www|\.com', case=False, regex=True).astype(int)
    df['has_number']    = df['message'].str.contains(r'\d', regex=True).astype(int)
    df['has_currency']  = df['message'].str.contains(
                            r'\u00a3|\$|\u20ac|free|win|prize|cash|claim|offer',
                            case=False, regex=True).astype(int)
    df['caps_ratio']    = (df['message'].str.count(r'[A-Z]') /
                           (df['message'].str.len() + 1))
    df['exclaim_count'] = df['message'].str.count(r'!')
    return df

df = add_meta_features(df)
print('Meta features sample:')
print(df[META_COLS].head())

## 6. Train / Test Split

In [ ]:
X_text = df['message_clean']
X_meta = df[META_COLS]
y      = df['label']

X_train_t, X_test_t, X_train_m, X_test_m, y_train, y_test = train_test_split(
    X_text, X_meta, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f'Train: {len(y_train)} samples | Test: {len(y_test)} samples')
print(f'Spam in train: {y_train.sum()} ({y_train.mean()*100:.1f}%)')

## 7. TF-IDF Vectorisation (Improved)

| Setting | Baseline | Improved |
|---|---|---|
| `max_features` | 5 000 | **8 000** |
| `ngram_range` | (1,2) | **(1,3)** |
| `sublinear_tf` | False | **True** |

In [ ]:
vectorizer = TfidfVectorizer(
    stop_words='english',
    max_features=8000,
    ngram_range=(1, 3),
    sublinear_tf=True
)

X_train_vec = vectorizer.fit_transform(X_train_t)
X_test_vec  = vectorizer.transform(X_test_t)

print('Vocab size:', len(vectorizer.vocabulary_))
print('Train matrix shape:', X_train_vec.shape)

## 8. Combine TF-IDF + Meta Features

In [ ]:
scaler = MinMaxScaler()
X_train_meta_scaled = csr_matrix(scaler.fit_transform(X_train_m))
X_test_meta_scaled  = csr_matrix(scaler.transform(X_test_m))

X_train_full = hstack([X_train_vec, X_train_meta_scaled])
X_test_full  = hstack([X_test_vec,  X_test_meta_scaled])

print('Combined train matrix shape:', X_train_full.shape)

## 9. Model Comparison

Naive Bayes requires non-negative input, so it uses TF-IDF only.

In [ ]:
models = {
    'SVM (linear, balanced)': SVC(kernel='linear', C=0.5, probability=True,
                                   random_state=42, class_weight='balanced'),
    'Logistic Regression':    LogisticRegression(C=1.0, max_iter=500,
                                   class_weight='balanced', random_state=42),
    'Naive Bayes':            MultinomialNB(alpha=0.1),
}

results = {}
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, model in models.items():
    use_full = (name != 'Naive Bayes')
    Xtr = X_train_full if use_full else X_train_vec
    Xte = X_test_full  if use_full else X_test_vec

    cv_scores = cross_val_score(model, Xtr, y_train, cv=skf, scoring='f1')
    model.fit(Xtr, y_train)
    pred = model.predict(Xte)

    results[name] = {
        'model':      model,
        'cv_f1_mean': cv_scores.mean(),
        'cv_f1_std':  cv_scores.std(),
        'accuracy':   accuracy_score(y_test, pred),
        'f1':         f1_score(y_test, pred),
        'precision':  precision_score(y_test, pred),
        'recall':     recall_score(y_test, pred),
        'preds':      pred,
    }
    print(f'{name}')
    print(f'  CV F1 : {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}')
    print(f'  Test  : acc={accuracy_score(y_test, pred):.4f}  '
          f'f1={f1_score(y_test, pred):.4f}  '
          f'prec={precision_score(y_test, pred):.4f}  '
          f'rec={recall_score(y_test, pred):.4f}')
    print()

## 10. Comparison Chart

In [ ]:
metrics = ['accuracy', 'f1', 'precision', 'recall']
model_names = list(results.keys())
vals = {m: [results[n][m] for n in model_names] for m in metrics}

x = np.arange(len(model_names))
width = 0.2

fig, ax = plt.subplots(figsize=(12, 5))
for i, metric in enumerate(metrics):
    ax.bar(x + i * width, vals[metric], width, label=metric.capitalize())

ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(model_names, fontsize=11)
ax.set_ylim(0.85, 1.01)
ax.set_ylabel('Score')
ax.set_title('Model Comparison — Test Set Metrics')
ax.legend()
plt.tight_layout()
plt.show()

## 11. Best Model — Detailed Report

In [ ]:
best_name = max(results, key=lambda n: results[n]['f1'])
best = results[best_name]
print(f'Best model: {best_name}  (F1 = {best["f1"]:.4f})')
print()
print(classification_report(y_test, best['preds'], target_names=['Ham','Spam']))

cm = confusion_matrix(y_test, best['preds'])
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Ham','Spam'], yticklabels=['Ham','Spam'])
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.title(f'Confusion Matrix — {best_name}')
plt.tight_layout()
plt.show()

## 12. Save Artifacts

> **Note:** `svm_spam_model.pkl` always contains the **best** model (by F1).  
> `meta_scaler.pkl` is new — the Streamlit UI also loads it.  
> `model_results.json` lets the UI show the comparison table without re-training.

In [ ]:
best_model_obj = results[best_name]['model']
joblib.dump(best_model_obj, 'svm_spam_model.pkl')
joblib.dump(vectorizer,     'tfidf_vectorizer.pkl')
joblib.dump(scaler,         'meta_scaler.pkl')

summary = {name: {k: round(v, 4) for k, v in r.items()
                  if k not in ('model', 'preds')}
           for name, r in results.items()}
with open('model_results.json', 'w') as f:
    json.dump(summary, f, indent=2)

print('Saved:')
print('  svm_spam_model.pkl   — best model:', best_name)
print('  tfidf_vectorizer.pkl')
print('  meta_scaler.pkl      — NEW (for meta features in Streamlit)')
print('  model_results.json   — NEW (comparison table for Streamlit)')

## 13. Quick Inference Test

In [ ]:
test_messages = [
    "Congratulations! You've WON a 1000 Walmart gift card. Click here to claim now!!!",
    "Hey, are you coming to the meeting tomorrow at 10?",
    "FREE entry to win tickets. Call 0800 now!",
    "Can you pick up some milk on your way home?",
]

def predict_message(msg):
    cleaned   = clean_text(msg)
    tfidf_vec = vectorizer.transform([cleaned])
    meta_row  = pd.DataFrame([{
        'msg_len':       len(msg),
        'word_count':    len(msg.split()),
        'has_url':       int(bool(re.search(r'http|www|\.com', msg, re.I))),
        'has_number':    int(bool(re.search(r'\d', msg))),
        'has_currency':  int(bool(re.search(r'free|win|prize|cash|claim|offer', msg, re.I))),
        'caps_ratio':    sum(1 for c in msg if c.isupper()) / (len(msg) + 1),
        'exclaim_count': msg.count('!'),
    }])
    meta_scaled = csr_matrix(scaler.transform(meta_row))
    use_full    = (best_name != 'Naive Bayes')
    vec_input   = hstack([tfidf_vec, meta_scaled]) if use_full else tfidf_vec
    pred        = best_model_obj.predict(vec_input)[0]
    label       = 'SPAM' if pred == 1 else 'HAM'
    if hasattr(best_model_obj, 'predict_proba'):
        prob = best_model_obj.predict_proba(vec_input)[0]
        conf = prob[pred]
        return f'[{label}] ({conf*100:.1f}% confidence) — {msg[:70]}'
    return f'[{label}] — {msg[:70]}'

for msg in test_messages:
    print(predict_message(msg))